## Authenticate to Hugging Face

In [14]:
import huggingface_hub #required !pip install huggingface_hub
#Login to Hugging Face
huggingface_hub.login()

# Huggingface Text Classification

NOTE: A GPU is needed on Google Colab, go to Runtime -> Change runtime type -> Harware Accelerator -> GPU

What's going to be covered in this project:
1. Getting / creating a text dataset with Hugging Face Datasets
2. Preparing text data for machine learning (tokenization)
3. Creating and training a text classification with Hugging Face Transformers
4. Uploading our trained model to the Hugging Face Hub
5. Turning our model into a demo with Gradio + Hugging Face Spaces

## 2. Import necessary libraries

In [8]:
import transformers

In [9]:
# Install dependencies (this is mostly for Google Colab)
try:
  import datasets, evaluate, accelerate
  import gradio as gr
except ModuleNotFoundError:
  !pip install -U datasets evaluate accelerate gradio -q
  import datasets, evaluate, accelerate
  import gradio as gr

import random
import numpy as np
import pandas as pd
import torch
import transformers

print(f"Using transformers version: {transformers.__version__}")
print(f"Using datasets version: {datasets.__version__}")
print(f"Using torch version: {torch.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.8 MB/s eta 0:00:00
Using transformers version: 5.0.0
Using datasets version: 4.0.0
Using torch version: 2.10.0+cu128


## 3. Getting the Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset('mrdbourke/learn_hf_food_not_food_image_captions')
dataset

In [ ]:
# Get features
dataset.column_names

In [ ]:
# Access the training split
dataset['train'][0]

In [ ]:
# Inspect Random Samples
random_index = random.randint(0,len(dataset['train']))
dataset['train'][random_index]

In [ ]:
# Check count for each label
from collections import Counter

Counter(dataset['train']['label'])

### Convert the dataset to a DataFrame

In [ ]:
food_not_food_df = pd.DataFrame(dataset['train'])
food_not_food_df.head()

In [ ]:
# Value counts
food_not_food_df.label.value_counts()

## 4. Preparing data for text classification

1. Tokenize text - turn text to numbers
2. Create a split

In [ ]:
# Create mappings programmatically from dataset
id2label = {id: label for id, label in enumerate(dataset["train"].unique("label")[::-1])}
label2id = {label: id for id, label in enumerate(dataset["train"].unique("label")[::-1])}
id2label, label2id

In [ ]:
# Turn labels into 0 or 1
def map_labels_to_number(df):
  df["label"] = label2id[df["label"]]
  return df

sample = {
    "text": "This is a sample",
    "label": "not_food"
}

In [ ]:
# Test on the sample
map_labels_to_number(sample)

In [ ]:
# Map the dataset with the above function
dataset = dataset["train"].map(map_labels_to_number)
dataset[:5]

## Split the dataset into training and test sets

1. Train set = Model will learn patterns on this dataset
2. Validation set (optional) = We can tune our model's hyperparameter on this set
3. Test set = Model will evaluate patterns on this dataset

In [ ]:
# Creating train and test sets
dataset = dataset.shuffle()
dataset = dataset.train_test_split(test_size=0.2, seed=42)
dataset

# Tokenization

The `transformers` library has in-built support for Hugging Face `tokenizers`.
And the class `transformers.AutoTokenizer` helps pair a model to a tokenizer.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path="distilbert/distilbert-base-uncased",
                                          use_fast=True) # use fast implementation (on by default, note: this requires Rust installed)
tokenizer

* `input_ids` = Text turned into numbers
* `attention_mask` = Whether or not to pay attention to certain tokens (1 = Yes; pay attention, 0 = No; don't pay attention

In [ ]:
tokenizer("I love ice-cream", "What is life?")

In [ ]:
# Get the length of tokenizer vocab
length_of_tokenizer = len(tokenizer.vocab)
print(f"[INFO] Number of items in our tokenizer vocab: {length_of_tokenizer}")

# Get the maximum sequence length the tokenizer can handle
max_sequence_length = tokenizer.model_max_length
print(f"[INFO] Maximum sequence length: {max_sequence_length}")

In [ ]:
tokenizer.vocab["nepal"]

In [ ]:
# Try to tokenize an emoji
tokenizer.convert_ids_to_tokens(tokenizer("🍕").input_ids)

In [ ]:
# Get the first 5 items in the tokenizer vocab
print(list(tokenizer.vocab.items())[:5])

## Making a preprocessing function to tokenize text

Make it easy to go from sample -> tokenized_sample

In [ ]:
dataset['train']

In [ ]:
def tokenize_input(examples):
  from transformers import AutoTokenizer
  """
    Tokenize given example text and return the tokenized text.
  """
  return tokenizer(
      examples["text"],
      padding=True,   # Pad short sequences to longest sequence length in batch
      truncation=True # Truncate long sequences to the maximum sequence length
  )


In [ ]:
# Test the function
example_sample = {"text": "I love pizza", "label": 1}
print(tokenize_input(example_sample))

In [ ]:
# Test with a long sequence
long_text = ''.join(chr(random.choice([32] + list(range(97,123)))) for _ in range(1000))

In [ ]:
example_sample_2 = {'text': long_text, 'label': 0}
tokenized_long_text = tokenize_input(example_sample_2)
len(tokenized_long_text['input_ids'])

In [ ]:
# Map our tokenized text function to the dataset
tokenized_dataset = dataset.map(function=tokenize_input,
                                batched=True, # set batched=True to tokenize across batches of samples at a time rather than individually
                                batch_size=1000)
tokenized_dataset

In [ ]:
# Get two samples from the tokenized datasets
train_tokenized_sample = tokenized_dataset["train"][0]
test_tokenized_sample = tokenized_dataset["test"][0]

for key in train_tokenized_sample.keys():
  print(f"[INFO] Key: {key}")
  print(f"[INFO] Train Sample: {train_tokenized_sample[key]}")
  print(f"[INFO] Test Sample: {test_tokenized_sample[key]}")
  print()

### Tokenization Takeaways

1. **Tokenizers** = Turn data into numbers (e.g. text -> map to number)
2. Many models are out there and have different tokenizers, HuggingFace's `Auto` classes (e.g. `AutoTokenizer`, `AutoProcessor`, `AutoModel` etc help to match tokeniezrs to models)
3. Tokenization can happen in parallel using `map` and batched functions

## Evaluation Metrics

Evaluation metrics provide a numerical measure of how well a model performs.

Common evaluation metrics for classification:

1. **Accuracy**  
   Percentage of predictions the model got correct out of all predictions.  
   Accuracy = `(TP + TN) / (TP + TN + FP + FN)`

2. **Precision**  
   Out of all predictions the model labeled as positive, how many were actually positive.  
   Precision = `TP / (TP + FP)`

3. **Recall (Sensitivity / True Positive Rate)**  
   Out of all actual positive examples, how many the model correctly identified.  
   Recall = `TP / (TP + FN)`

4. **F1 Score**  
   Harmonic mean of Precision and Recall.  
   F1 = `2 × (Precision × Recall) / (Precision + Recall)`

In [ ]:
# Import evaluation modules
import evaluate
import numpy as np
from typing import Tuple

accuracy_metric = evaluate.load("accuracy")

def compute_accuracy(predictions_and_labels: Tuple[np.array, np.array]):
  """
    Computes the accuracy of the model by comparing the predictions and labels.
  """
  predictions, labels = predictions_and_labels

  return accuracy_metric.compute(predictions=predictions, references=labels)


In [ ]:
# Example predictions and accuracy score
example_preds_all_correct = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
example_preds_one_incorrect = np.array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0])
example_labels = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

# Test the function
print(f"Accuracy 1: {compute_accuracy((example_preds_one_incorrect, example_labels))}")
print(f"Accuracy 2: {compute_accuracy((example_preds_all_correct, example_labels))}")

## Model Training

* For this project, we'll leverage transfer learning, it is a technique, unique to deep learning models that enables us to use the patterns one models has learned on a particular context, and use it on another context

#### Why Transfer Learning?
- It can leverage an existing neural network architecture proven to work on problems similar to our own.
- It can leverage a working network architecture which has already learned patterns on similar data to our own (often results in greater results with less data).

Hence, The model performs better than from scratch.

### Workflow:
- Create and Preprocess Data - Done
- Define the model we'd like to use for our problem.
- Define training arguments for training our model `transformers.TrainingArguments`.
  * These are also known as "hyperparameters" = settings on your model that you can adjust
  * Parameters = weights / patterns in the model that gets updated automatically
- Pass `TrainingArguments` to an instance of `transformers.Trainer` class.
- Train the model by calling `Training.train()`.
- Save the model (to our local machine or to the Huggin Face Hub).
- Evaluate the trained model by making and inspecting predictions on the test data.
- Turn the model into a shareable demo.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path='distilbert/distilbert-base-uncased',
    num_labels=2, # classify into food, not_food
    id2label=id2label,
    label2id=label2id
)

In [ ]:
model

### Model Parts:
1. `embeddings` - Embeddings are a form of learned representation of tokens. So if tokens are a direct mapping from token to number, then embeddings are a learned vector representation.
2. `transformer` - Model architecture backbone, this has discovered patterns/relationships in the embeddings.
3. `classifier` - We need to customize this layer to suit our problem.

## NOTE:
*If getting input errors from passing a sample to a model, make sure the sample you pass to your model is formatting in the same way your model was trained on. For example, if your model used a specific tokenizer, make sure to tokenize your text before passing it to the model.*

### Count the parameters in our model

**Weights/parameters** = Small numeric opportunities for a model to learn patterns in data.

In [ ]:
# Count Parameters
def count_params(model):
  """
    Count the parameters of a PyTorch model.
  """
  trainable_parameters = sum(param.numel() for param in model.parameters() if param.requires_grad) # numel() = Number of elements, if requires_grad is True, then the parameter
                                                                                                   # is updated during trainig
  total_parameters = sum(param.numel() for param in model.parameters())

  return {"trainable_parameters": trainable_parameters, "total_parameters": total_parameters}

In [ ]:
count_params(model)

Looks like our model has around 67M parameters and **all** of them are trainable.

NOTE:
* Generally, the more parameters a model has, the more capacity it has to learn.
* For comparison models such las Llama 3 8B has 8 billion parameters.
* If you want the best possible performance, generally more parameters is better.
  * However, with more parameters requires more compute + time.
  * You'll be surprised how well a smaller model can perform with specific data.

### Creating a folder to save your model

In [4]:
# Create model output directory
from pathlib import Path

# Create models dir
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

# Create model save name
model_save_name = "learn_hf_food_not_food_text_classifier-distilbert-base-uncased"

# Create model save path
model_save_dir = Path(models_dir, model_save_name)
model_save_dir

PosixPath('models/learn_hf_food_not_food_text_classifier-distilbert-base-uncased')

## Setting up training arguments (hyperparameters) with TrainingArugments

In [ ]:
from transformers import TrainingArguments

print(f"[INFO] Saving model checkpoints: {model_save_dir}")

BATCH_SIZE = 32

# Create training arguments
training_args = TrainingArguments(
    output_dir=model_save_dir,
    learning_rate=0.0001,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    use_cpu=True,
    seed=42,
    load_best_model_at_end=True,
    logging_strategy="epoch",  # Log at each epoch
    report_to="none",  # This will save the results to other third party platforms like, weights & biases.
    push_to_hub=False, # This will save the model directly to the huggingface hub, once the training finishes.
    hub_private_repo=False # When uploading to Hugging Face Hub, do you want your repo to be private or public? (default: public)
)


### NOTE: FIXING THE ACCURACY FUNCTION
The accuracy function above doesn't work for Trainer.train() as it as a bug in it which makes the output of accuracy in the wrong format to what Trainer.train() expects.

In [ ]:
# Sample code to debug accuracy
input_predictions = np.array([[-1.216908, 1.2238812]])
input_references = np.array([0])

In [ ]:
# Import evaluation modules
import evaluate
import numpy as np
from typing import Tuple

accuracy_metric = evaluate.load("accuracy")

def compute_accuracy2(predictions_and_labels: Tuple[np.array, np.array]):
  """
    Computes the accuracy of the model by comparing the predictions and labels.
  """
  predictions, labels = predictions_and_labels

  if len(predictions.shape) > 1:      # This snippet will make sure the predictions is a scalar value, which is the maximum value that
    predictions = np.argmax(predictions, axis=1)


  return accuracy_metric.compute(predictions=predictions, references=labels)


In [ ]:
compute_accuracy2((input_predictions, input_references))

In [ ]:
# Small demonstration showing how argmax works:
import numpy as np

predictions = np.array([
    [2.3, 0.1],
    [0.4, 1.7],
    [3.2, 0.2]
])

print(np.argmax(predictions, axis=1))   # Pick the maximum value index from each row, axis = 1; means look across columns.

In [ ]:
from transformers import Trainer

# Setup Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    compute_metrics=compute_accuracy2
)
trainer

In [ ]:
results = trainer.train()

In [ ]:
results

In [ ]:
results.metrics

In [ ]:
# Inspect training metrics
for key, value in results.metrics.items():
  print(f"{key}: {value}")


### Save the model

In [ ]:
# Save the model
print(f"[INFO] Saving model to {model_save_dir}")
trainer.save_model(output_dir=model_save_dir)

### Inspect the model training metrics

In [ ]:
# Get training history
trainer_history_all = trainer.state.log_history
trainer_history_metrics = trainer_history_all[:-1]
tainer_history_training_time = trainer_history_all[-1]

# View the first three
trainer_history_metrics[:3]

In [ ]:
import pprint

# Extract eval and training metrics
trainer_history_training_set = [entry for entry in trainer_history_metrics if "loss" in entry]
trainer_history_eval_set = [entry for entry in trainer_history_metrics if "eval_loss" in entry]

pprint.pprint(f'Training set metrics: {trainer_history_training_set[:2]}')
pprint.pprint(f'Validation set metrics: {trainer_history_eval_set[:2]}')

## Visualising the loss curves

In [ ]:
# Plotting the loss curves using matplotlib
import matplotlib.pyplot as plt

# Extract losses
training_loss = [entry["loss"] for entry in trainer_history_training_set]
validation_loss = [entry["eval_loss"] for entry in trainer_history_eval_set]

# Epochs
training_epochs = [entry["epoch"] for entry in trainer_history_training_set]
validation_epochs = [entry["epoch"] for entry in trainer_history_eval_set]

# Histogram for Loss Curves
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(training_epochs, training_loss, label="Training Loss")
ax.plot(validation_epochs, validation_loss, label="Validation Loss")

ax.set_title("Training vs Validation Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()

plt.show()

### Converting our list to DataFrame

In [ ]:
trainer_history_training_df = pd.DataFrame(trainer_history_training_set)
trainer_history_eval_df = pd.DataFrame(trainer_history_eval_set)

print(trainer_history_training_df.head(2))
print(trainer_history_eval_df.head(2))

## Pushing the model to HugginFace Hub

Why?
- So the model can be shared within the team for testing.
- The history of the model can be tracked along with its performance.

To write to HuggingFace:
1. Either setup the token with "read and write" access
2. Or, setup `huggingface-cli`

To save the model to Hugging Face Hub, use `Trainer.push_to_hub` method

In [ ]:
# Save the model to HF Hub
model_upload_url = trainer.push_to_hub(
    commit_message='Uploading text classifier (food/not_food) model'
)
print(f"[INFO]: Model uploaded to {model_upload_url}")

In [ ]:
print("Hello world")

### Making and evaluating predictions on unseen data
**NOTE:** Evaluating model is important as training a model.

In [ ]:
# Predict using the model
predictions_all = trainer.predict(tokenized_dataset["test"])
predictions_values = predictions_all.predictions
predictions_metrics = predictions_all.metrics

In [ ]:
print(f"[INFO] Prediction metrics on the test data: {predictions_metrics}")

In [ ]:
# Convert predicted logits -> predictions probabilities using torch.softmax, which is the predicted labels

import torch
from sklearn.metrics import accuracy_score

# 1. Get prediction probabilities using torch.softmax
pred_probs = torch.softmax(torch.tensor(predictions_values), dim=1)

# 2. Get the predicted labels
pred_labels = torch.argmax(pred_probs, dim=1)
pred_labels

In [ ]:
# Compute accuracy
accuracy = accuracy_score(y_true=tokenized_dataset["test"]["label"], y_pred=pred_labels)
print(f"Accuracy score: {accuracy}")

### Exploring our models predictions probabilities

It's a very good way to evaluate a model by sorting predictions by prediction probabilities and seeing where the model went wrong.

In [ ]:
# Make a DataFrame of test preds
test_predictions_df = pd.DataFrame({
    "text": tokenized_dataset["test"]["text"],
    "true_label": tokenized_dataset["test"]["label"],
    "pred_label": pred_labels,
    "pred_probs": torch.max(pred_probs, dim=1).values
})

In [ ]:
# Predictions
test_predictions_df.head()

In [ ]:
# Show the lowest predictions probability
test_predictions_df.sort_values(by="pred_probs", ascending=True).head()

### Making and inspecting predictions on custom text data

#### Ways to make predictions (inference)
1. **Pipeline mode** - Using `transformers.pipeline` to load our model and perform text classification.

2. **PyTorch mode** - Using a combination of `transformers.AutoTokenizer` and `transformers.AutoModelForSequenceClassification` and passing each our target model name.

### Each mode supports:
1. Predictions one at a time (fast but can be slower with many many samples).
2. Batches of predictions at a time (faster but up to a point, e.g., say you predict on 32 samples at a time, this may be way faster than one at a time but if you go to 128 samples at a time, you may not see many more speedups).

In [5]:
# Local model path:
local_model_path = Path(models_dir, "learn_hf_food_not_food_text_classifier-distilbert-base-uncased")

# Hugging Face model path
huggingface_model_path = 'AnubhavKarki/learn_hf_food_not_food_text_classifier-distilbert-base-uncased'

In [ ]:
# Setup for our device to make predictions
# NOTE: Generally the faster the hardware accelerator, the faster the predictions.
# For example if you have a dedicated GPU, you should use it over CPU.

def set_device():
  if torch.cuda.is_available():
    device = torch.device("cuda")
  elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
  else:
    device = torch.device("cpu")
  return device

In [ ]:
# Get device name
DEVICE = set_device()
DEVICE

### Making predictions with Pipeline mode

In [ ]:
# Import libraries
import torch
from transformers import pipeline

# Set Batch size
BATCH_SIZE = 32 # REMINDER: Predictions speed often increases with higher batch size (e.g., 1 -> 32 but can saturate at even higher batch sizes)

# Create an instance of transformers.pipeline
food_not_food_classifier = pipeline(
    task='text-classification',
    model=local_model_path,
    device=DEVICE,
    top_k=1,
    batch_size=BATCH_SIZE
)

In [ ]:
custom_text_1 = "Homage to Catalonia is written by George Orwell."
custom_text_2 = "Salmon and Rice is a healthy food combination."

print(food_not_food_classifier(custom_text_2))

In [ ]:
# Delete pipeline to create a new one
del pipeline

In [ ]:
from transformers import pipeline

# Use pipeline with a model from Hugging Face
food_not_food_classifier_hf = pipeline(
    task='text-classification',
    model=huggingface_model_path ,
    device=DEVICE,
    top_k=1,
    batch_size=BATCH_SIZE
)


In [ ]:
# Create a list of sentences to make predictions on
sentences = [
    "I whipped up a fresh batch of code, but it seems to have a syntax error.",
    "We need to marinate these ideas overnight before presenting them to the client.",
    "The new software is definitely a spicy upgrade, taking some time to get used to.",
    "Her social media post was the perfect recipe for a viral sensation.",
    "He served up a rebuttal full of facts, leaving his opponent speechless.",
    "The team needs to simmer down a bit before tackling the next challenge.",
    "The presentation was a delicious blend of humor and information, keeping the audience engaged.",
    "A beautiful array of fake wax foods (shokuhin sampuru) in the front of a Japanese restaurant.",
    "Daniel Bourke is really cool :D",
    "My favoruite food is biltong!"
]

In [ ]:
# Predict on array of inputs
food_not_food_classifier_hf(sentences)

### Time the model across larger sample sizes

In [ ]:
import time

# Create 1000 sentences
sentences_1000 = sentences * 100

In [ ]:
# Time how long it takes to make predictions on all sentences (one at a time)
print(f"[INFO] Number of sentences: {len(sentences_1000)}")
start_time = time.time()
for sentence in sentences_1000:
  food_not_food_classifier(sentence)
end_time = time.time()

print(f"Total time taken: {end_time - start_time:.6f}")

In [ ]:
# Time how long it takes to make predicitons on 32 sentences at a time
print(f"[INFO] Number of sentences: {len(sentences_1000)}")
start_time = time.time()
for i in range(0, len(sentences_1000), BATCH_SIZE):
  food_not_food_classifier(sentences_1000[i:i+BATCH_SIZE])
end_time = time.time()

print(f"Total time taken: {end_time - start_time:.10f}")

### Using pipeline in Batches

In [ ]:
for i in [10, 100, 1000, 10_000]:
  sentences_big = sentences * i
  print(f"[INFO] Number of sentences: {len(sentences_big)}")

  start_time = time.time()

  # Predict on all sentences in batch mode
  food_not_food_classifier(sentences_big)

  end_time = time.time()
  print(f"Total time taken: {end_time - start_time:.10f}")
  avg_time_per_sentence_batch_mode = (end_time - start_time)/ len(sentences_big)
  print(f"Average time per sentence: {avg_time_per_sentence_batch_mode}")

### Performing Inference with PyTorch
Steps with PyTorch predictions:

1. Create the tokenizer with `AutoTokenizer`.
2. Create the model with `AutoModel` (`AutoModelForSequenceClassification`)
3. Tokenize text with 1
4. Make predictions with 2
5. Format prediction

In [ ]:
from transformers import AutoTokenizer

# Setup model path
model_path = "AnubhavKarki/learn_hf_food_not_food_text_classifier-distilbert-base-uncased"

# Create an example to predict on
sample_food_text = "A delicious photo of a plate of scrambled eggs, toast, and bacon."

# Prepare the tokenizer
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_path)
inputs = tokenizer(sample_food_text, return_tensors="pt")
inputs

In [ ]:
from transformers import AutoModelForSequenceClassification

# Create the model
model = AutoModelForSequenceClassification.from_pretrained(pretrained_model_name_or_path=model_path)
model

In [ ]:
import torch

model.eval()
# with torch.no_grads():
# OR,
with torch.inference_mode():
  outputs = model(**inputs)
  print(f"[INFO] Outputs: {outputs}")
  logits = outputs.logits
  print(f"[INFO] Logits: {logits}")

In [ ]:
# Convert logits to prediction probability + label
predicted_class_id = np.argmax(outputs.logits).item()
predicted_probability_score = torch.softmax(outputs.logits, dim=1).max().item()

In [ ]:
predicted_probability_score

## Putting it all together

Let's go end to end.

From data import to model building to model evaluation to inference and finally saving.

In [ ]:
# 1. Import necessary libraries
import pprint
from pathlib import Path

import numpy as np
import torch

import datasets
import evaluate

from transformers import pipeline
from transformers import Trainer, TrainingArguments
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 2. Setup variables for model training and saving pipeline
DATASET_NAME = "mrdbourke/learn_hf_food_not_food_image_captions"
MODEL_NAME = "distilbert/distilbert-base-uncased"
MODEL_SAVE_DIR_NAME = "models/learn_hf_food_not_food_text_classifier-distilbert-base-uncased"

# 3. Create a directory for saving models
print(f"[INFO] Creating a model directory")
models_save_dir = Path("models")
models_save_dir.mkdir(parents=True, exist_ok=True)

# 4. Load and preprocess the dataset from Hugging Face
print(f"[INFO] Loading the dataset from Hugging Face")
dataset = datasets.load_dataset(path=DATASET_NAME)

print(f"[INFO] Mapping labels to class")
id2label = {id: label for id, label in enumerate(dataset["train"].unique("label")[::-1])}
label2id = {label: id for id, label in enumerate(dataset["train"].unique("label")[::-1])}

# Create a function to map IDs to labels in the dataset
def map_ids_to_labels(batch):
  batch["labels"] = label2id[batch["label"]]
  return batch

# Map our preprocessing function to our dataset
dataset = dataset["train"].map(map_ids_to_labels)

print(f"[INFO] Setting up an evaluation metric")
accuracy_metric = evaluate.load("accuracy")

# Split the dataset
dataset = dataset.train_test_split(test_size=0.2, seed=42)

# 5. Import a tokenizer and map it to our dataset.
print(f"[INFO] Importing a tokenizer")
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=MODEL_NAME, use_fast=True)

# Create function to tokenize text samples (turn text into numbers)
def tokenize_text(examples):
  return tokenizer(
      examples["text"],
      padding=True,
      truncation=True
)
tokenize_dataset = dataset.map(tokenize_text, batched=True, batch_size=1000)
print(tokenized_dataset["train"][:5])

# 6. Set up an evaluation metric
def compute_accuracy(predictions_and_labels: Tuple[np.array, np.array]):
  """
    Computes the accuracy of the model by comparing the predictions and labels.
  """
  predictions, labels = predictions_and_labels

  if len(predictions.shape) > 1:      # This snippet will make sure the predictions is a scalar value, which is the maximum value that
    predictions = np.argmax(predictions, axis=1)

  return accuracy_metric.compute(predictions=predictions, references=labels)

# 7. Set up a model
model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path=MODEL_NAME,
    num_labels=2, # classify into food, not_food
    id2label=id2label,
    label2id=label2id
)

# 8. Train the model
print(f"[INFO] Training the model")
print(f"[INFO] Saving model checkpoints: {MODEL_SAVE_DIR_NAME}")

BATCH_SIZE = 32

# Create training arguments
training_args = TrainingArguments(
    output_dir=model_save_dir,
    learning_rate=0.0001,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    use_cpu=True,
    seed=42,
    load_best_model_at_end=True,
    logging_strategy="epoch",  # Log at each epoch
    report_to="none",  # This will save the results to other third party platforms like, weights & biases.
    push_to_hub=False, # This will save the model directly to the huggingface hub, once the training finishes.
    hub_private_repo=False # When uploading to Hugging Face Hub, do you want your repo to be private or public? (default: public)
)

# Setup Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    compute_metrics=compute_accuracy
)

# Start training
results = trainer.train()

# 9. Save the trained model (to a local directory)
print(f"[INFO] Saving the model to {MODEL_SAVE_DIR_NAME}")
trainer.save_model(output_dir=MODEL_SAVE_DIR_NAME)

# 10. Push the model to the Hugging Face Hub
print(f"[INFO] Pushing the model to Hugging Face Hub")
model_upload_url = trainer.push_to_hub(
    commit_message='Uploading text classifier (food/not_food) model'
)

# 11. Evaluate the model on the test data
print(f"[INFO] Performing evaluation on the test dataset...")
predictions_all = trainer.predict(tokenized_dataset["test"])
prediction_values = predictions_all.predictions
prediction_metrics = predictions_all.metrics

print(f"[INFO] Prediction metrics on the test data:")
pprint.pprint(prediction_metrics)

### Turning our model into a demo

Now, the model can be turned into a demo app that helps us easily share it with others so they can try it out.

Tools Used: Gradio; It helps to create the workflow: inputs -> model/function -> outputs

# Create a function to perform inference

1. Take an input of string
2. Setup a text classification pipeline
3. Get the output from the pipeline
4. Return the ouput from the pipeline in step 3 as a dictionary with the format `{"Label_1": prob_1, "label_2": prob_2}`

In [18]:
from typing import Dict

# 1. Create a function that takes string input
def food_not_food_classifier(text: str) -> Dict[str, float]:
  # 2. Setup food not food classifier
  food_not_food_classifier = pipeline(task="text-classification",
                                      model=local_model_path,
                                      batch=32,
                                      device='cuda' if torch.cuda.is_available() else "cpu",
                                      top_k=None
                            )
  # 3. Get the outputs from the pipeline
  outputs = food_not_food_classifier(text)[0]

  # 4. Format output for Gradio
  output_dict = {output["label"]: output["score"] for output in outputs}

  return output_dict

In [19]:
pred = food_not_food_classifier(text="Yo we're building a local demo!")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [20]:
pred

{'not_food': 0.9983370304107666, 'food': 0.0016629802994430065}

### Building a small Gradio demo to run locally

In [21]:
# 1. Import gradio
import gradio as gr

# 2. Create gradio Interface
demo = gr.Interface(
    fn=food_not_food_classifier,
    inputs="text",
    outputs=gr.Label(num_top_classes=2),
    title='Food / Not Food CLassifier',
    description = "A text classifier to detemine if a sentence is about food or not.",
    examples=[
        "My code is going into production",
        "I have a recipe to cook delicious tiramisu"
      ]
)

# 3. Launch the interface
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e3a266039c5221fd14.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Making the demo publicly accessible
There are two main ways with Hugging Face Spaces:
1. Manually - We can go to huggingface.co/spaces -> "Create new space" -> add our files and publish.
2. Programmatically - We can use the HuggingFace Hub Python API and add our files to Spaces with code.

To create a space programmatically we're going to three files:
1. `app.py`: This is the main app with functionality of our demo.
2. `requirements.txt`: These are the dependencies which our app will require.
3. `README.md`: This will explain what our project/demo is about. And will also add some metadata in YAML format.

In [22]:
# Make a directory to store the demo
from pathlib import Path

demo_dir = Path("../demos")
demo_dir.mkdir(exist_ok=True)

# Create a folder for the food_not_food_text_classifier demo
food_not_food_classifier_demo_dir = Path(demo_dir, 'food_not_food_text_classifier')
food_not_food_classifier_demo_dir.mkdir(exist_ok=True)

### Making an app.py file

The `app.py` will contain the main logic for the application to run.
When we upload it to Hugging Face Spaces, Spaces will automatically try to run `app.py`

In `app.py` file:
1. Import packages
2. Define function to use the model (this will work with Gradio).
3. Create a demo with Gradio
4. Run the demo with `demo.launch()`.

In [24]:
%%writefile ../demos/food_not_food_text_classifier/app.py
import torch
import gradio as gr
from typing import Dict
from transformers import pipeline

# Define the function to use with the model
# 1. Create a function that takes string input
def food_not_food_classifier(text: str) -> Dict[str, float]:
  # 2. Setup food not food classifier
  food_not_food_classifier = pipeline(task="text-classification",
                                      model=huggingface_model_path,
                                      batch=32,
                                      device='cuda' if torch.cuda.is_available() else "cpu",
                                      top_k=None
                                    )
  # 3. Get the outputs from the pipeline
  outputs = food_not_food_classifier(text)[0]

  # 4. Format output for Gradio
  output_dict = {output["label"]: output["score"] for output in outputs}

  return output_dict

# Create a Gradio Interface
description = """
A text classifier to determine if a sentence is about food or not food.

Fine-tuned from [DistilBERT] (https://huggingface.co/distilbert/distilbert-base-uncased) a [dataset of LLM generated food/not_food image captions].
"""

demo = gr.Interface(
    fn=food_not_food_classifier,
    inputs='text',
    outputs=gr.Label(num_top_classes=2),
    title='Food / Not Food Classifier',
    description=description,
    examples=[
        "My code is going into production",
        "I have a recipe to cook delicious tiramisu"
    ]
)

# 4. Launch the interface
if __name__ == "__main__":
  demo.launch()

Overwriting ../demos/food_not_food_text_classifier/app.py


### Making a README file

This file is in markdown format.
With a special YAML block at the top.
The YAML block at the top is used for metadata + settings.

In [25]:
%%writefile ../demos/food_not_food_text_classifier/README.md

---
title: Food Not Food Text Classifier
colorForm: blue
colorTo: yellow
sdk: gradio
sdk_version: 4.40.0
app_file: app.py
pinned: true
license: apache-2.0
---

# Food Not Food Text Classifier

Small demo to showcase a text classifier to determine if a sentence is about food or not food.
DistilBERT model fine-tuned on a small synthetic dataset of [250 generated food/not_food image captions].


Writing ../demos/food_not_food_text_classifier/README.md


### Making a requirements file

This file is going to tell Hugging Face Spaces which versions, which packages to use.

If we don't create this file, we may get error for missing packages.

In [26]:
%%writefile ../demos/food_not_food_text_classifier/requirements.txt

gradio==4.40.0
torch
transformers

Writing ../demos/food_not_food_text_classifier/requirements.txt


### Uploading the demo to Hugging Face Spaces

Tools Used: Hugging Face Hub Python API

1. Import necessary functions
2. Define the upload
3. Create a repository
4. Get the name of the repository from the upload
5. Upload the contents of model to Hugging Face Hub
6. Inspect the results

In [29]:
# 1. Import libraries
from huggingface_hub import (
    create_repo,
    get_full_repo_name,
    upload_folder,
    upload_file
)

# 2. Define the parameters we'd like to use for uploading our Space
LOCAL_DEMO_DIR = "../demos/food_not_food_text_classifier"
HF_TARGET_SPACE_NAME = "food_not_food_text_classifier"
HF_REPO_TYPE = "space"
HF_SPACE_SDK = "gradio"

# 3. Create a Space repository on Hugging Face Hub
print(f"[INFO] Creating space repository on HF Hub")
create_repo(
    repo_id=HF_TARGET_SPACE_NAME,
    repo_type=HF_REPO_TYPE,
    private=False,
    space_sdk=HF_SPACE_SDK,
    exist_ok=True
)

# 4. Get the full repo name
hf_full_repo_name = get_full_repo_name(model_id=HF_TARGET_SPACE_NAME)
print(f"[INFO] Full repo name: {hf_full_repo_name}")

# 5. Upload our demo folder
print(f"[INFO] Uploading our demo to Hugging Face Hub")
folder_upload_url = upload_folder(
    repo_id=hf_full_repo_name,
    folder_path=LOCAL_DEMO_DIR,
    path_in_repo=".",
    repo_type=HF_REPO_TYPE,
    commit_message="Uploading food/not food text classifier demo from the video!"
)

print(f"[INFO] Demo folder successfully uploaded with commit URL: {folder_upload_url}")

[INFO] Creating space repository on HF Hub
[INFO] Full repo name: AnubhavKarki/food_not_food_text_classifier
[INFO] Uploading our demo to Hugging Face Hub
[INFO] Demo folder successfully uploaded with commit URL: https://huggingface.co/spaces/AnubhavKarki/food_not_food_text_classifier/commit/c20eeca856525ae8926b00898f2b53d3d3ef909f


### Embedding the Hugging Face Space

In [ ]:
from IPython.display import HTML
HTML(data='''
                # <- Add your <iframe> here, I have removed mine for security purpose.
     ''')